# Convert Raw .txt for each GB to a single cleaned CSV file

This notebook takes the raw SI data provided by Guanying in the PSED Google Drive and converts it into a cleaner format.

In [31]:
import os

import pandas as pd

In [32]:
# iterate through subfolders of data/raw_si_data
cleaned_data = []
index = []

raw_data_dir = 'data/raw_si_data'
for subfolder in os.listdir(raw_data_dir):
    subfolder_path = os.path.join(raw_data_dir, subfolder)
    if os.path.isdir(subfolder_path):
        # find all text files in subfolder or its subfolders
        for root, dirs, files in os.walk(subfolder_path):
            for file in files:
                if file.endswith('.txt'):
                    file_path = os.path.join(root, file)
                    # read the text file into a dataframe
                    df = pd.read_csv(file_path, delimiter=' ', header=None, skiprows=1)
                    # all text files have 2 columns, Location and Thermal Resistance
                    # and add then to cleaned_data
                    df.columns = ['Location', 'Thermal Resistance']
                    # get rid of index
                    df.reset_index(drop=True, inplace=True)
                    cleaned_data.append(df)
                    # add file path with underscores instead of slashes to index
                    underscores = file_path.replace('/', '_').replace('\\', '_')
                    # they will all look like `data_raw_si_data_XXX_M42result_RB_along_GB.txt`,
                    # just grab the XXX part
                    part = underscores.split('_')[5:-4]
                    index.append('_'.join(part))

# concatenate all dataframes in cleaned_data into a single dataframe
final_df = pd.concat(cleaned_data, keys=index)
# get rid of second column (just the default integer index for each dataframe)
final_df.reset_index(level=1, drop=True, inplace=True)
# save final_df to data/cleaned_si_data/cleaned_si_data.csv
final_df.to_csv('data/cleaned_si_data/cleaned_si_data.csv')